<a href="https://colab.research.google.com/github/aounraza379/SecureMed-Fed/blob/main/securemed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random

# Set the number of data points (simulating about 15-16 minutes of data)
data_points = 1000

# Normal heart rates (60 to 90 beats per minute)
heart_rates = [random.randint(60, 90) for _ in range(data_points)]

# Creating a "Cardiac Anomaly" (Emergency)
# Changing the last 50 readings to be dangerously high
for i in range(950, 1000):
    heart_rates[i] = random.randint(150, 180)

# Timestamp for each reading
timestamps = pd.date_range(start="2026-04-01 08:00:00", periods=data_points, freq='S')

# Spreadsheet (DataFrame)
df = pd.DataFrame({
    'Timestamp': timestamps,
    'Heart_Rate': heart_rates
})

# Save it as a CSV file
df.to_csv('patient_data.csv', index=False)

print("File 'patient_data.csv' has been created.")
print(df.tail(10))

File 'patient_data.csv' has been created.
              Timestamp  Heart_Rate
990 2026-04-01 08:16:30         170
991 2026-04-01 08:16:31         173
992 2026-04-01 08:16:32         159
993 2026-04-01 08:16:33         164
994 2026-04-01 08:16:34         167
995 2026-04-01 08:16:35         151
996 2026-04-01 08:16:36         164
997 2026-04-01 08:16:37         159
998 2026-04-01 08:16:38         170
999 2026-04-01 08:16:39         159


/tmp/ipykernel_1751/2463331751.py:17: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  timestamps = pd.date_range(start="2026-04-01 08:00:00", periods=data_points, freq='S')


In [2]:
# Load csv
data = pd.read_csv('patient_data.csv')

# Safety Threshold (120 BPM is the danger zone for an elderly person at rest)
THRESHOLD = 120
alert_triggered = False

print("--- SecureMed Watchdog Scanning Started ---")

# Scan 'Heart_Rate' column
for index, row in data.iterrows():
    current_bpm = row['Heart_Rate']
    timestamp = row['Timestamp']

    if current_bpm > THRESHOLD:
        print(f"ALERT! High Heart Rate Detected: {current_bpm} BPM at {timestamp}")
        alert_triggered = True

if alert_triggered:
    print("\n--- FINAL STATUS: CARDIAC ANOMALY DETECTED. EMERGENCY SERVICES NOTIFIED. ---")
else:
    print("\n--- FINAL STATUS: Patient is Stable. ---")

--- SecureMed Watchdog Scanning Started ---
ALERT! High Heart Rate Detected: 179 BPM at 2026-04-01 08:15:50
ALERT! High Heart Rate Detected: 158 BPM at 2026-04-01 08:15:51
ALERT! High Heart Rate Detected: 172 BPM at 2026-04-01 08:15:52
ALERT! High Heart Rate Detected: 173 BPM at 2026-04-01 08:15:53
ALERT! High Heart Rate Detected: 163 BPM at 2026-04-01 08:15:54
ALERT! High Heart Rate Detected: 152 BPM at 2026-04-01 08:15:55
ALERT! High Heart Rate Detected: 163 BPM at 2026-04-01 08:15:56
ALERT! High Heart Rate Detected: 160 BPM at 2026-04-01 08:15:57
ALERT! High Heart Rate Detected: 150 BPM at 2026-04-01 08:15:58
ALERT! High Heart Rate Detected: 170 BPM at 2026-04-01 08:15:59
ALERT! High Heart Rate Detected: 163 BPM at 2026-04-01 08:16:00
ALERT! High Heart Rate Detected: 164 BPM at 2026-04-01 08:16:01
ALERT! High Heart Rate Detected: 161 BPM at 2026-04-01 08:16:02
ALERT! High Heart Rate Detected: 161 BPM at 2026-04-01 08:16:03
ALERT! High Heart Rate Detected: 174 BPM at 2026-04-01 08:16

In [3]:
import numpy as np
import hashlib

def generate_secure_update(heart_rate_column):
    """
    Simulates a Local Differential Privacy update for Federated Learning.
    Masks raw data before transmission to the central server.
    """
    # Calculate local statistics
    local_mean = heart_rate_column.mean()
    local_std = heart_rate_column.std()

    # Add Gaussian Noise (Differential Privacy) to mask the exact mean
    noise = np.random.normal(0, 0.1, 1)[0]
    masked_update = local_mean + noise

    # Generate a unique hash for the device to ensure data integrity (ZKP Foundation)
    device_id = "Device_Elderly_001"
    data_hash = hashlib.sha256(str(masked_update).encode()).hexdigest()

    return {
        "masked_mean": round(masked_update, 2),
        "integrity_hash": data_hash,
        "status": "Anomaly_Detected" if local_mean > 110 else "Normal"
    }

# Execute Local Privacy Masking
local_data = pd.read_csv('patient_data.csv')
secure_packet = generate_secure_update(local_data['Heart_Rate'])

print(f"--- Transmission Packet Created ---")
print(f"Masked Mean (Shared): {secure_packet['masked_mean']}")
print(f"Integrity Hash (Encrypted): {secure_packet['integrity_hash']}")
print(f"System Status: {secure_packet['status']}")

--- Transmission Packet Created ---
Masked Mean (Shared): 79.2
Integrity Hash (Encrypted): b0f5f53322993125441bff1d633806b453424f77a02723907282916995002159
System Status: Normal


In [4]:
import hashlib
import time

def generate_emergency_proof(heart_rate, secret_key="PATIENT_SECRET_001"):
    """
    Simulates a Zero-Knowledge Proof (ZKP) for emergency alerts.
    Verifies the severity of the event without exposing raw vitals.
    """
    timestamp = str(int(time.time()))

    # A "Commitment": A cryptographic hash of the data + a secret key
    # This acts as a 'digital seal' that cannot be altered.
    commitment = hashlib.sha256(f"{heart_rate}{secret_key}{timestamp}".encode()).hexdigest()

    # Define the 'Proof' condition (Is it a real emergency?)
    is_emergency = heart_rate > 120

    # Create the 'Proof' packet for the hospital
    # We send the commitment and the status, but NOT the secret_key or the exact heart_rate
    proof_packet = {
        "commitment": commitment,
        "is_emergency": is_emergency,
        "timestamp": timestamp,
        "verification_status": "PENDING"
    }

    return proof_packet

def verify_emergency_proof(proof_packet, original_commitment):
    """
    Simulates the server-side verification of the cryptographic seal.
    """
    if proof_packet["commitment"] == original_commitment:
        proof_packet["verification_status"] = "VERIFIED_AUTHENTIC"
    else:
        proof_packet["verification_status"] = "INVALID_PROOF"

    return proof_packet

# Execute ZKP Logic for a sample anomaly
emergency_proof = generate_emergency_proof(165)
verified_result = verify_emergency_proof(emergency_proof, emergency_proof["commitment"])

print(f"--- Zero-Knowledge Proof (ZKP) Verification ---")
print(f"Emergency Status: {verified_result['is_emergency']}")
print(f"Cryptographic Seal: {verified_result['commitment']}")
print(f"Verification Result: {verified_result['verification_status']}")

--- Zero-Knowledge Proof (ZKP) Verification ---
Emergency Status: True
Cryptographic Seal: 0f26a76aa5c945ad1bd9179ca2996a924999ec70753dae3351949be5acdc946d
Verification Result: VERIFIED_AUTHENTIC
